# Speech And Egg Channel Analysis

Separate and inspect speech and electroglottographic channels in a supplied recording.

Stage: prototype.


In [ ]:
import librosa
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt

waveform, sample_rate = librosa.load(r'/content/SOUND.wav', sr=None, mono=False)

print("Shape of waveform array:", waveform.shape)

speech_signal = waveform[0]
egg_signal = waveform[1]

sf.write('speech_signal.wav', speech_signal, sample_rate)
sf.write('egg_signal.wav', egg_signal, sample_rate)

speech_file = 'speech_signal.wav'
speech_signal, fs = librosa.load(speech_file, sr=None)
egg_file = 'egg_signal.wav'
egg_signal, _ = librosa.load(egg_file, sr=None)
differentiated_egg = np.diff(egg_signal)

fs = 16000
frame_length = int(0.1 * fs)
hop_length = frame_length // 2

zcr = librosa.feature.zero_crossing_rate(speech_signal, frame_length=frame_length, hop_length=hop_length)
voiced_indices = np.where(zcr > 0.05)

random_voiced_index = np.random.choice(voiced_indices[1])
voiced_region_start = random_voiced_index * hop_length
voiced_region_end = voiced_region_start + frame_length

voiced_speech = speech_signal[voiced_region_start:voiced_region_end]

voiced_egg = egg_signal[voiced_region_start:voiced_region_end]

voiced_differentiated_egg = differentiated_egg[voiced_region_start:voiced_region_end-1]  # One sample less due to differentiation

threshold = 0.005
peaks = np.where(voiced_differentiated_egg > threshold)[0]
troughs = np.where(voiced_differentiated_egg < -threshold)[0]

time = np.arange(len(voiced_speech)) / fs

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(time, voiced_speech, color='red')
plt.title('Speech Signal (16 kHz)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(time, voiced_egg, color='blue')
plt.title('EGG Signal (16 kHz)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(time[:-1], voiced_differentiated_egg, color='green')  # One sample less due to differentiation
plt.scatter(time[:-1][peaks], voiced_differentiated_egg[peaks], color='red', label='GCI')
plt.scatter(time[:-1][troughs], voiced_differentiated_egg[troughs], color='blue', label='GOI')
plt.title('Differentiated EGG Signal (16 kHz) with Marked GCIs and GOIs')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

gci_time_diffs = np.diff(peaks) / fs

if len(gci_time_diffs) > 0:
    average_gci_time_diff = np.mean(gci_time_diffs)
    pitch = 1 / average_gci_time_diff
    print("Pitch (F0) for one glottal cycle:", pitch, "Hz")
else:
    print("No GCIs found.")

speed_quotient = (troughs[-1] - troughs[0]) / (peaks[-1] - peaks[0])
close_quotient = (peaks[-1] - troughs[0]) / (peaks[-1] - peaks[0])
print("Speed Quotient (SQ):", speed_quotient)
print("Close Quotient (CQ):", close_quotient)

min_length = min(len(gci_time_diffs), len(goi_time_diffs))

jitter = np.mean(np.abs(gci_time_diffs[:min_length] - goi_time_diffs[:min_length]))

peak_amplitudes = voiced_differentiated_egg[peaks]
trough_amplitudes = voiced_differentiated_egg[troughs]
min_length = min(len(peak_amplitudes), len(trough_amplitudes))
shimmer = np.mean(np.abs(peak_amplitudes[:min_length] - trough_amplitudes[:min_length]))
print("Jitter:", jitter)
print("Shimmer:", shimmer)
